# LiDAR 스캔 기록 (손으로 밀며 기록)

로봇을 손으로 천천히 밀면서 LiDAR 스캔을 30초 동안 파일(`lidar_scan_1.npz`)로 저장합니다. 다시 기록할 때는 코드 셀의 `OUT` 번호만 바꿉니다(`lidar_scan_2.npz` …).

## 하는 일
- 모터를 IDLE로 두어 바퀴를 손으로 밀 수 있게 합니다.
- 한 바퀴(회전)씩 묶어 각도·거리를 기록합니다. 유효 점이 200개 미만인 회전은 버립니다.
- 스캔 시작이 실패하면(`Descriptor length mismatch`) 최대 5번 재시도합니다. 원인은 확인하지 못했습니다.
- 기록 시간은 첫 측정값이 들어온 시점부터 잽니다(재시도 시간 제외).
- 파일에는 `t`(회전 시작 시각), `offsets`(회전별 점 경계), `angle_deg`, `dist_mm`가 들어갑니다.

## 기록하기 전에 (실측 방법)
- 로봇의 같은 지점(예: 바퀴 중심선)을 시작과 끝에서 기준으로 삼고, 그 지점 바로 아래 바닥에 테이프를 붙입니다.
- 가능한 한 직선으로 밀고, 중간에 뒤로 밀리거나 멈췄다 다시 가지 않습니다.
- 손이나 몸이 LiDAR 옆에 있으면 로봇과 함께 움직이는 점이 기록됩니다. 로봇 뒤쪽을 밀거나 막대를 써서 몸을 LiDAR에서 멀리 둡니다.
- 양옆이 평행한 벽인 복도에서는 앞뒤 이동을 구할 수 없으므로, 앞이나 뒤 6m 안에 끝 벽이나 물체가 보이게 둡니다.

## 실행 결과
30초 동안 301회 회전 저장, 회전당 평균 387점, 약 10회전/초.

## 실측 (줄자)
- 시작·끝 위치를 바닥에 테이프로 표시하고 잰 이동 거리: 151.5 cm (기준: 바퀴가 바닥에 닿는 지점).
- 같은 로그를 LiDAR 스캔 매칭으로 추정한 이동 거리: 약 146 cm (줄자 대비 −3.8%). 한 번의 기록이라 정확도를 단정할 수 없습니다.

In [3]:
import time
import numpy as np
from rplidar import RPLidar, RPLidarException
from tiki.mini import TikiMini

PORT = '/dev/ttyUSB0'
BAUDRATE = 460800
RECORD_SEC = 30            # 기록 시간(초)
MIN_POINTS = 200           # 한 바퀴에서 유효한 점이 이보다 적으면 버림
OUT = 'lidar_scan_1.npz'

# 손으로 밀 수 있도록 모터를 IDLE로 둠 (IDLE에서 바퀴를 손으로 돌릴 수 있었음)
robot = TikiMini()
robot.set_motor_mode(robot.MOTOR_MODE_IDLE)

def measures_with_retry(lidar, retries=5):
    for attempt in range(1, retries + 1):
        gen = lidar.iter_measures()
        try:
            first = next(gen)
        except RPLidarException as e:
            print(f"스캔 시작 실패 ({attempt}/{retries}): {e}")
            try:
                lidar.stop()
            except Exception:
                pass
            time.sleep(1.0)
            continue
        yield first
        yield from gen
        return
    raise RuntimeError("LiDAR 스캔 시작에 계속 실패했습니다")

t_list, offsets, ang, dist = [], [0], [], []
cur_a, cur_d, cur_t = [], [], None
started = False

lidar = RPLidar(PORT, baudrate=BAUDRATE, timeout=2)
input(f"준비되면 Enter. 그 다음 {RECORD_SEC}초 동안 로봇을 손으로 아주 천천히 밀어 주세요")
t_start = None            # 첫 측정값이 들어올 때부터 시간을 잼 (시작 재시도 시간 제외)
last_print = None
try:
    for new_scan, quality, angle, distance in measures_with_retry(lidar):
        now = time.time()
        if t_start is None:
            t_start = last_print = now
        if now - t_start > RECORD_SEC:
            break
        if new_scan:
            if started and len(cur_a) >= MIN_POINTS:
                t_list.append(cur_t)
                ang.extend(cur_a)
                dist.extend(cur_d)
                offsets.append(len(ang))
            started = True
            cur_a, cur_d, cur_t = [], [], now
        if started and distance > 0:
            cur_a.append(angle)
            cur_d.append(distance)
        if now - last_print >= 5:
            print(f"{now - t_start:4.0f}s  저장된 회전 {len(t_list)}회")
            last_print = now
except KeyboardInterrupt:
    print("중단됨 - 지금까지의 데이터를 저장합니다")
finally:
    t_end = time.time()
    for fn in (lidar.stop, lidar.stop_motor, lidar.disconnect):
        try:
            fn()
        except Exception:
            pass
    if t_list:
        np.savez_compressed(OUT,
                            t=np.array(t_list, dtype=np.float64),
                            offsets=np.array(offsets, dtype=np.int64),
                            angle_deg=np.array(ang, dtype=np.float32),
                            dist_mm=np.array(dist, dtype=np.float32))
        n = len(t_list)
        print(f"\n저장: {OUT}  회전 {n}회, 회전당 평균 {len(ang) / n:.0f}점, "
              f"약 {n / (t_end - t_start):.1f}회전/초")
    else:
        print("저장할 데이터가 없습니다")


준비되면 Enter. 그 다음 30초 동안 로봇을 손으로 아주 천천히 밀어 주세요 


   5s  저장된 회전 50회
  10s  저장된 회전 100회
  15s  저장된 회전 151회
  20s  저장된 회전 201회
  25s  저장된 회전 251회

저장: lidar_scan_1.npz  회전 301회, 회전당 평균 387점, 약 10.0회전/초
